In [ ]:
# Params
#=========================================================
# KNMI data
param_stationID = '260'
param_from_date = '1996'
param_to_date = '2025' 
param_vars = "average_temperature, minimum_temperature, maximum_temperature, total_precipitation, average_windspeed, average_winddirection, relative_humidity, minimum_humidity, maximum_humidity"

param_stationID_spinup = '260'
param_from_date_spinup = '1896'
param_to_date_spinup = '1996' 
param_vars_spinup = "average_temperature, minimum_temperature, maximum_temperature, total_precipitation, average_windspeed, average_winddirection, relative_humidity, minimum_humidity, maximum_humidity"

# Ecoregion map 
param_ecoregion_name = '101'

# Climate generator 
param_climate_time_series = 'Monthly_SequencedYears'
param_spinup_climate_time_series = 'Monthly_SequencedYears'

# Biomass initialization file 
param_timestep = '10'

# Scenario file 
param_duration = '100'
param_cell_length = '100'

In [ ]:
# Install and load all packages
#=========================================================

if (!require("pacman")) install.packages("pacman")
pacman::p_load("dplyr", "tidyr", "stringr", "lubridate", "readr", "Hmisc", "terra", "sf", "raster", "processx")


In [ ]:
# KNMI data retriever
#=========================================================
# Create directory
if(!dir.exists("/tmp/data")) dir.create("/tmp/data")
#=========================================================
# Part 1: Set up for KNMI daily data 
#=========================================================
# Parameter validation - check whether all parameters were filled in by user
if (param_stationID    == "" ||
    param_from_date    == "" ||
    param_to_date      == "" ||
    length(param_vars) == 0) {
  stop("All parameters must be provided by the user.")
}

# Split climate variables to make columns 
param_vars <- unlist(strsplit(param_vars, ",\\s*"))

param_vars[param_vars == "average_windspeed"]     <- "FG"
param_vars[param_vars == "average_temperature"]   <- "TG"
param_vars[param_vars == "minimum_temperature"]   <- "TN"
param_vars[param_vars == "maximum_temperature"]   <- "TX"
param_vars[param_vars == "total_precipitation"]   <- "RH"
param_vars[param_vars == "average_winddirection"] <- "DDVEC"
param_vars[param_vars == "relative_humidity"]     <- "UG"
param_vars[param_vars == "minimum_humidity"]      <- "UN"
param_vars[param_vars == "maximum_humidity"]      <- "UX"

#=========================================================
# Part2: Set up for KNMI spinup data 
#=========================================================
# Parameter validation - check whether all parameters were filled in by user
if (param_stationID_spinup    == "" ||
    param_from_date_spinup    == "" ||
    param_to_date_spinup      == "" ||
    length(param_vars_spinup) == 0) {
  stop("All parameters must be provided by the user.")
}

#Split climate variables to make columns and rename so that function can use set parameter values 
param_vars_spinup <- unlist(strsplit(param_vars_spinup, ",\\s*"))

param_vars_spinup[param_vars_spinup == "average_windspeed"]     <- "FG"
param_vars_spinup[param_vars_spinup == "average_temperature"]   <- "TG"
param_vars_spinup[param_vars_spinup == "minimum_temperature"]   <- "TN"
param_vars_spinup[param_vars_spinup == "maximum_temperature"]   <- "TX"
param_vars_spinup[param_vars_spinup == "total_precipitation"]   <- "RH"
param_vars_spinup[param_vars_spinup == "average_winddirection"] <- "DDVEC"
param_vars_spinup[param_vars_spinup == "relative_humidity"]     <- "UG"
param_vars_spinup[param_vars_spinup == "minimum_humidity"]      <- "UN"
param_vars_spinup[param_vars_spinup == "maximum_humidity"]      <- "UX"

#=========================================================
# Part3: Write function to get the KNMI data, this includes checks whether the dates are filled in correctly and logically 
# Does not require API because it is historic weather data 
#=========================================================
get_data <- function(stationID, from, to, vars) {
  
  # Date parsing - checking whether date is logical 
  if (!is.character(from) ||
      !is.character(to) ||
      stringr::str_length(from) %% 2 == 1 ||
      stringr::str_length(to) %% 2 == 1) {
    stop(
      "The values for 'from' and 'to' must be strings in the format 'YYYY', 'YYYYMM' or 'YYYYMMDD'."
    )
  }
  
  if (stringr::str_length(from) == 6) {
    from_date <- paste0(from, "01")
  } else if (stringr::str_length(from) == 4) {
    from_date <- paste0(from, "0101")
  } else {
    from_date <- from
  }
  
  if (stringr::str_length(to) == 8) {
    to_date <- to
  } else {
    if (stringr::str_length(to) == 6) {
      to <- paste0(to, "01")
    } else if (stringr::str_length(to) == 4) {
      to <- paste0(to, "1231")
    }
    
    to_date <- lubridate::ymd(to) %>%
      lubridate::ceiling_date(unit = "month") - 1
    
    to_date <- gsub("-", "", as.character(to_date))
  }
  
  if (as.numeric(to_date) < as.numeric(from_date)) {
    stop("'from' must be earlier than or equal to 'to'")
  }
  
  # Station validation - check whether station can be selected. The list contains all the KNMI weather stations that collect daily data 
  selectable_stations <- c("260", "275", "209", "210", "215", "225", "229", "235", "240", "242", "248", "249", "251", "257", "258", "260", "265", 
                           "267", "269", "270", "273", "275", "277", "278", "279", "280", "283", "285", "286", "290", "308", "310", "311", "312", 
                           "313", "315", "316", "319", "323", "324", "330", "331", "340", "343", "344", "348", "350", "356", "370", "375", "377",
                           "380", "391", "392")  
  stationID <- as.character(stationID)
  
  if (!stationID %in% selectable_stations) {
    stop(
      paste0(
        "Invalid stationID. Choose one of: ",
        paste(selectable_stations, collapse = ", ")
      ))
  }
  
  # Build URL to retrieve the KNMI data from the website 
  baseURL <- "https://www.daggegevens.knmi.nl/klimatologie/daggegevens"
  URL <- paste0(
    baseURL,
    "?start=", from_date,
    "&end=", to_date,
    "&stns=", stationID,
    "&ALL"
  )
  
  # Download data from KNMI 
  data_daily <- read_csv(
    URL,
    col_names = FALSE,
    comment = "#",
    show_col_types = FALSE
  ) %>%
    dplyr::as_tibble()
  
    colnames(data_daily) <-
      c(
        "STN", "YYYYMMDD", "DDVEC", "FHVEC", "FG", "FHX", "FHXH", "FHN", "FHNH", "FXX", "FXXH", "TG", "TN", "TNH", "TX", "TXH", "T10N",
        "T10NH", "SQ", "SP", "Q", "DR", "RH", "RHX", "RHXH", "PG", "PX", "PXH", "PN", "PNH", "VVN", "VVNH", "VVX", "VVXH", "NG",
        "UG", "UX", "UXH", "UN", "UNH", "EV24"
      )
    
  # Select only requested variables as indicated when parameters were set 
  data_daily <- data_daily %>%
    #tidyr::drop_na(dplyr::all_of(vars)) %>%
    dplyr::select(all_of(c("YYYYMMDD", vars)))
  
  # Scaling because KNMI works in 0.1
  if ("FG" %in% vars) data_daily$FG <- data_daily$FG / 10
  if ("TG" %in% vars) data_daily$TG <- data_daily$TG / 10
  if ("TN" %in% vars) data_daily$TN <- data_daily$TN / 10
  if ("TX" %in% vars) data_daily$TX <- data_daily$TX / 10
  if ("RH" %in% vars) data_daily$RH <- data_daily$RH / 10
  
  return(data_daily)
}

#=========================================================
# Part 4: Run the functions twice for both daily and spinup 
#=========================================================
KNMI_daily <- get_data(
    stationID = param_stationID,
    from      = param_from_date,
    to        = param_to_date,
    vars      = param_vars
)

KNMI_spinup <- get_data(
  stationID = param_stationID_spinup,
  from      = param_from_date_spinup,
  to        = param_to_date_spinup,
  vars      = param_vars_spinup
)

# Save the output as csv to tmp data to pass onto next cells 
KNMI_daily_datafile <- "/tmp/data/KNMI_daily.csv"
write.csv(KNMI_daily, 
          file = KNMI_daily_datafile, 
          row.names = FALSE)

KNMI_spinup_datafile <- "/tmp/data/KNMI_spinup.csv"
write.csv(KNMI_spinup, 
          file = KNMI_spinup_datafile, 
          row.names = FALSE)

In [ ]:
# KNMI data preprocessor
#=========================================================
# Read the file that was retrieved in previous cell 
KNMI_daily  <- read.csv(KNMI_daily_datafile)
KNMI_spinup <- read.csv(KNMI_spinup_datafile)

# Part 1: Preprocess the KNMI daily data 
# Reordering the date from YYYYMMDD to Year, Month, and Day in separate columns
Preprocessed_KNMI_daily <- KNMI_daily %>%
  dplyr::mutate(YYYYMMDD = as.character(YYYYMMDD)) %>%
  dplyr::mutate(
    date  = ymd(YYYYMMDD),
    Year  = year(date),
    Month = month(date),
    Day   = day(date)
  ) %>%
  dplyr::select(-YYYYMMDD, -date) %>%

# Renaming KNMI variables to match LANDIS-II names & reorder and rename columns 
  dplyr::rename(any_of(c(
    windSpeed = "FG",
    temp      = "TG",
    mintemp   = "TN",
    maxtemp   = "TX",
    precip    = "RH",
    RH        = "UG",
    maxRH     = "UX",
    minRH     = "UN",
    windDirection = "DDVEC"
  ))) %>%
tidyr::pivot_longer(
    cols = -c(Year, Month, Day),
    names_to = "Variable",
    values_to = param_ecoregion_name
  ) %>%
  dplyr::select(Year, Month, Day, Variable, {{param_ecoregion_name}}) %>%
  tidyr::drop_na({{param_ecoregion_name}})

# Part 2: Preprocess the KNMI spinup data 
# Reordering the date from YYYYMMDD to Year, Month, and Day 
Preprocessed_KNMI_spinup <- KNMI_spinup %>%
 dplyr::mutate(YYYYMMDD = as.character(YYYYMMDD)) %>%
  dplyr::mutate(
    date  = ymd(YYYYMMDD),
    Year  = year(date),
    Month = month(date),
    Day   = day(date)
  ) %>%
  dplyr::select(-YYYYMMDD, -date) %>%

# Renaming KNMI variables to match LANDIS-II names & order and rename columns 
  dplyr::rename(any_of(c(
    windSpeed = "FG",
    temp      = "TG",
    mintemp   = "TN",
    maxtemp   = "TX",
    precip    = "RH",
    RH        = "UG",
    maxRH     = "UX",
    minRH     = "UN",
    windDirection = "DDVEC"      
  ))) %>%
tidyr::pivot_longer(
  cols = -c(Year, Month, Day),
  names_to = "Variable",
  values_to = param_ecoregion_name  
) %>%
  dplyr::select(Year, Month, Day, Variable, {{param_ecoregion_name}}) %>%
  tidyr::drop_na({{param_ecoregion_name}})

# Part 3: Aggregate daily KNMI data to monthly values
Preprocessed_KNMI_daily[[param_ecoregion_name]] <- 
  as.numeric(Preprocessed_KNMI_daily[[param_ecoregion_name]])

KNMI_monthly <- Preprocessed_KNMI_daily %>%
  dplyr::group_by(Year, Month, Variable) %>%
  dplyr::summarise(
    value = ifelse(
      first(Variable) == "precip",
      sum(.data[[param_ecoregion_name]], na.rm = TRUE),
      mean(.data[[param_ecoregion_name]], na.rm = TRUE)
    ),
    .groups = "drop"
  ) %>%
  dplyr::rename(!!param_ecoregion_name := value)

# Also for the spinup data
Preprocessed_KNMI_spinup[[param_ecoregion_name]] <- 
  as.numeric(Preprocessed_KNMI_spinup[[param_ecoregion_name]])

KNMI_monthly_spinup <- Preprocessed_KNMI_spinup %>%
  dplyr::group_by(Year, Month, Variable) %>%
  dplyr::summarise(
    value = ifelse(
      first(Variable) == "precip",
      sum(.data[[param_ecoregion_name]], na.rm = TRUE),
      mean(.data[[param_ecoregion_name]], na.rm = TRUE)
    ),
    .groups = "drop"
  ) %>%
  dplyr::rename(!!param_ecoregion_name := value)

# Save the output as csv to the cloud storage so it can be inspected by user after preprocessing 
#Preprocessed_KNMI_daily_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_daily.csv"
#Preprocessed_KNMI_spinup_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_spinup.csv"
KNMI_monthly_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_monthly.csv"
KNMI_monthly_spinup_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_spinup_monthly.csv"

#write.csv(Preprocessed_KNMI_daily, 
        #  file = Preprocessed_KNMI_daily_datafile, 
         # quote = FALSE,
          # row.names = FALSE)

write.csv(KNMI_monthly, 
          file = KNMI_monthly_datafile, 
          quote = FALSE,
          row.names = FALSE)

write.csv(KNMI_monthly_spinup, 
          file = KNMI_monthly_spinup_datafile, 
          quote = FALSE,
          row.names = FALSE)

#write.csv(Preprocessed_KNMI_spinup, 
 #         file = Preprocessed_KNMI_spinup_datafile, 
  #        quote = FALSE,
   #       row.names = FALSE)